# Tokenizer

In this section, we are going to build a simple tokenizer that allows us to tokenize our sentences into smaller components called tokens and decode input tokens into the original string. Each token has its own unique identifier.

In the tokenizer, we have two main processes:
- **Encoding**: Chunking our sentences into smaller components called token. Each token will be represented with a unique identifier
<br>
<img height="400px" width="700px" src="images/tokenizer_encoding.png"/>
- **Decoder**: Converting token ids into the original sentence

In this notebook, we will build a tokenizer using an algorithm called **Byte Pair Encoding**, which is used in building many large language models such as ChatGPT or Gemini.


## Materials

- [Decoder-Only Transformers: The Workhorse of Generative LLMs](https://cameronrwolfe.substack.com/p/decoder-only-transformers-the-workhorse)

## Importing dataset

Now let's build our tokenizer vocabulary by training it with more text data

In [ ]:
urls = {
    "Education and the good life": "https://www.gutenberg.org/cache/epub/70302/pg70302.txt",
    "The School and Society": "https://www.gutenberg.org/cache/epub/53910/pg53910.txt",
    "What Is and What Might Be": "https://www.gutenberg.org/cache/epub/20555/pg20555.txt",
    "How we think": "https://www.gutenberg.org/cache/epub/37423/pg37423.txt",
    "The Reform of Education": "https://www.gutenberg.org/cache/epub/36762/pg36762.txt"
}

# Build a script that allows you to extract and construct training dataset

In [19]:
# Import book text dataset 
with open("dataset/how_we_think.txt", 'r', encoding='utf-8-sig') as f:
    lines = f.readlines()
    # Merge those lines together
    text = "".join(lines)
    

### Byte Pair Encoding

#### Materials
- [Byte Pair Encoding Hugging Face](https://www.youtube.com/watch?v=HEikzVL-lZU)

#### Core ideas

**Rule 1**: Do not split frequently used words into smaller subwords

**Rule 2**: Split the rare words into smaller, meaningful subwords

- Eg: "play" should not be split. "playing" should be split into "play" and "ing"

#### Benefits
1. "plays" and "playing" comes from the same root "play"
2. Some words have different root words but share the suffix part such as "classification" and "location" which both share "cation"

**BFE algorithm**: Most common pair of consecutive bytes of data is replaced with a byte that does not occur in the data

In [2]:
# Build a Tokenizer Class
class Tokenizer:
    def __init__(self):
        self.token_to_id = {'<start>' : 1, '<end_of_text>' : 2, ' ' : 3, '<unk>' : 4}
        self.id_to_token = {1 : '<start>', 2 : '<end_of_text>', 3 : ' ', 4: "<unk>"}
        self.vocab = set(['<start>', '<end_of_text>', '<unk>', ' '])
        self.bp_merges = {}

    def train(self, text):
        assert len(text) > 0, "You must input a non-empty text"
        text_characters = []
        for char in text:
            if char not in self.vocab:
                self.vocab.add(char)
                id = len(self.vocab)
                self.token_to_id[char] = id
                self.id_to_token[id] = char
            text_characters.append(char)

        while True:
            frequency = {}
            # Contruct frequency from the text_characters
            for i in range(1, len(text_characters)):
                pair = (text_characters[i-1], text_characters[i])
                if pair not in frequency:
                    frequency[pair] = 0
                frequency[pair] += 1
            if not frequency: break

            most_commmon_pair, occurence = max(frequency.items(), key = lambda item: item[1])
            if occurence > 1:
                new_token = ''.join(most_commmon_pair)
                self.vocab.add(new_token)
                id = len(self.vocab)
                self.token_to_id[new_token] = id
                self.id_to_token[id] = new_token
                self.bp_merges[most_commmon_pair] = id # id here is the rank for our pair
                # Merge those tokens inside the text_characters
                new_text_characters = []

                index = 0
                while (index < len(text_characters)):
                    if (text_characters[index] == most_commmon_pair[0] and index < len(text_characters) - 1 and text_characters[index+1] == most_commmon_pair[1]):
                        new_text_characters.append(new_token)
                        index += 2 # Skip the next character
                    else:
                        new_text_characters.append(text_characters[index])
                        index += 1
                text_characters = new_text_characters
            else:
                break

    def encode(self, text: str, add_special_tokens = True):
        assert self.bp_merges, "You must train your tokenizer first!"
        tokens = list(text)
        # Merge based on the rank that a pair was constructed
        while True:
            best_rank = float("inf")
            best_pair = None
            candidate_index = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i+1])
                rank = self.bp_merges.get(pair, None)
                if rank is not None and rank < best_rank:
                    best_pair = pair
                    best_rank = rank
                    candidate_index = i
            if best_pair is None: break
            # Merge pair with lowest rank
            tokens[candidate_index] = ''.join(best_pair)
            del tokens[candidate_index + 1]

        ids = [self.token_to_id.get(token, self.token_to_id['<unk>']) for token in tokens]
        if add_special_tokens:
            ids = [self.token_to_id["<start>"]] + ids + [self.token_to_id['<end_of_text>']]

        return ids

    def decode(self, inputs):
        string =  "".join([self.id_to_token.get(id, self.id_to_token[4]) for id in inputs])
        return string

In [ ]:
a = [i for i in range(400000)]


In [24]:
tokenizer = Tokenizer()
# Train tokenizer
tokenizer.train(text)

In [2]:
import pickle
import os
# Save tokenizer data
def save_tokenizer(tokenizer: Tokenizer):
    
    tokenizer_path = "tokenizer"
    if not os.path.isdir(tokenizer_path):
        os.mkdir(tokenizer_path)

    with open(f"{tokenizer_path}/token_to_id.pkl", 'wb') as f:
        pickle.dump(f, tokenizer.token_to_id)

    with open(f"{tokenizer_path}/id_to_token.pkl", "wb") as f:
        pickle.dump(f, tokenizer.id_to_token)

    with open(f"{tokenizer_path}/bp_merges.pkl", "wb") as f:
        pickle.dump(f, tokenizer.bp_merges)

def load_tokenizer(tokenizer_path = 'tokenizer'):
    with open(f"{tokenizer_path}/token_to_id.pkl", "wb") as f:
        tokenizer.token_to_id = pickle.load(f)

    with open(f"{tokenizer_path}/id_to_token.pkl", "wb") as f:
        tokenizer.id_to_token = pickle.load(f)

    with open(f"{tokenizer_path}/bp_merges.pkl", "wb") as f:
        tokenizer.bp_merges = pickle.load(f)

In [25]:
sentence = "The lesson here is under the"
ids = tokenizer.encode(sentence)
[tokenizer.id_to_token[id] for id in ids]

['<start>',
 'The ',
 'lesson ',
 'h',
 'ere is ',
 'under',
 ' the',
 '<end_of_text>']

In [27]:
# ids = [1, 59, 109, 2590, 1275, 66, 257, 25, 2]
tokenizer.decode(ids)

'<start>The lesson here is under the<end_of_text>'


### Positional encoding

- Implement positional encoding: [Link](https://kazemnejad.com/blog/transformer_architecture_positional_encoding/)

In [57]:
# Implement Positional Encoding Layer that takes in input and return the output with added position informatio
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, context_size):
        super().__init__()

        i = (torch.arange(d_model) // 2).view(1, -1)
        positions = torch.arange(context_size).view(-1, 1)

        self.encoding = positions / (10000 ** (2 * i / d_model))
        self.encoding[:, 0::2] = torch.sin(self.encoding[:, 0::2])
        self.encoding[:, 1::2] = torch.cos(self.encoding[:, 1::2])
        self.encoding = self.encoding.unsqueeze(0)
    def forward(self, X):
        # X with shape: B x N x d_model
        N = X.shape[1]
        print(self.encoding.shape)
        return X + self.encoding[:, :N, :]


In [14]:
positional_encoding = PositionalEncoding()

inputs = torch.randn(10, 100, 512)

positional_encoding(inputs).shape

torch.Size([10, 100, 512])

### Create input-target pair for training

In [4]:
from torch.utils.data import Dataset, DataLoader, random_split

class EducationDataset(Dataset):
    def __init__(self, text: str, tokenizer: Tokenizer, context_size = 1024):
        self.context_size = context_size
        self.tokenizer = tokenizer
        # Naive approach first
        self.X = []
        self.y = []
        
        tokens = self.tokenizer.encode(text)
        print("token len: ", len(tokens)) 
        if len(tokens) <= context_size:
            self.X.append(tokens[:len(tokens)])
            self.y.append(tokens[1:])
        else:
            for i in range(len(tokens) - context_size + 1):
                self.X.append(tokens[i:i + context_size])
                self.y.append(tokens[i+1: i + context_size+1])
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        x, y = self.X[index], self.y[index]
        return torch.tensor(x), torch.tensor(y)

In [49]:
len(tokenizer.vocab)

14164

In [ ]:
dataset = EducationDataset(text[:20000], tokenizer)

# Split into training and testing dataset
train_ratio = 0.8
test_ratio = 1 - train_ratio
train_size = int(len(dataset) * train_ratio)
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Number of train batches: ", len(train_loader))
print("Number of test batches: ", len(test_loader))




### MultiHead - Self-attention mechanism

- Implement self-attention with Q, K, V approach

→ Revise on how to implement Masked multi-head attention which prevents the model from attending to subsequent time step from the current time step


In [15]:
# Codeblock 14
BATCH_SIZE = 1
SEQ_LENGTH = 1024  # (1)
VOCAB_SIZE = 1000  # (2)

D_MODEL = 768
NUM_HEADS = 8  # (3)
HIDDEN_DIM = D_MODEL * 4  # (4)
N_LAYERS = 3
DROPOUT_RATE= 0.3

In [13]:
a = torch.randn(3, 5, 300)

linear = nn.Linear(300, 10)
linear(a).shape

torch.Size([3, 5, 10])

In [25]:
scores = torch.randn(3,3)
print("before masking: ")
print(scores)
mask = torch.triu(torch.ones(3, 3), diagonal=1).bool()
final_scores = scores.masked_fill(mask, float('-inf'))
print("after masking: ")
print(final_scores)

before masking: 
tensor([[-1.1493, -1.8712, -2.0942],
        [-0.6027,  0.6475,  1.7875],
        [-0.0640,  0.1971, -0.7022]])
after masking: 
tensor([[-1.1493,    -inf,    -inf],
        [-0.6027,  0.6475,    -inf],
        [-0.0640,  0.1971, -0.7022]])


In [27]:
nn.Softmax(dim = -1)(final_scores)

tensor([[1.0000, 0.0000, 0.0000],
        [0.2227, 0.7773, 0.0000],
        [0.3538, 0.4593, 0.1869]])

In [34]:
a = torch.randn(1, 3, 3)
b = torch.randn(1, 3, 2)

c = torch.matmul(a, b)
c

tensor([[[-0.5387, -1.1826],
         [ 0.8317,  1.0182],
         [ 0.7889,  1.4942]]])

In [33]:
a*b

tensor([[[ 0.3942, -1.7764,  2.5634],
         [-0.4283,  0.2090, -1.3099],
         [-0.7276, -0.1256,  0.3400]]])

In [66]:
import math
class Self_Attention(nn.Module):
    def __init__(self, d_model, d_k, d_v, masked = False):
        super().__init__()
        self.d_k = d_k
        self.W_Q = nn.Linear(d_model, d_k)
        self.W_K = nn.Linear(d_model, d_k)
        self.W_V = nn.Linear(d_model, d_v)
        self.masked = masked

    def forward(self, X):
        # X has shape: B x N x D_model
        N = X.shape[1]
        Q = self.W_Q(X)
        K = self.W_K(X)
        V = self.W_V(X) # B X N x D_V

        Z = Q @ K.permute(0, 2, 1) / math.sqrt(self.d_k)

        masked_matrix = torch.ones(N, N)
        masked_matrix = torch.triu(masked_matrix, diagonal=1) * (-1e8) 

        if self.masked:
            Z = Z + masked_matrix
        return torch.matmul(torch.softmax(Z, dim = -1), V)


class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.attentions = [Self_Attention(d_model, d_k, d_v, masked=True) for i in range(self.num_heads)]
        self.output = nn.Linear(num_heads * d_v, d_model)
    def forward(self, X):
        X = torch.cat([attention(X) for attention in self.attentions], dim = -1)
        return self.output(X)


class FFN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden_dim = input_dim * 4
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, input_dim)

    def forward(self, X):
        X = nn.GELU()(self.fc1(X))
        return self.fc2(X)


class GPT_Block(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads = 8, dropout_rate = 0.3):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.masked_attention = MaskedMultiHeadAttention(d_model, d_k, d_v, num_heads)
        self.layer_norm2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model)
        self.dropout = nn.Dropout(dropout_rate)
    def forward(self, X):
        X = self.dropout(self.masked_attention(self.layer_norm1(X))) + X
        X = self.dropout(self.ffn(self.layer_norm2(X))) + X
        return X


class MiniGPT(nn.Module):
    def __init__(self, Nx, d_model, num_heads, vocab_size, dropout_rate, context_size):
        super().__init__()
        assert d_model%num_heads == 0, "num_heads must be divided by d_model"
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_embedding = PositionalEncoding(d_model, context_size)
        d_k = d_v = d_model // num_heads
        self.gpt_blocks = nn.Sequential(
            *[GPT_Block(d_model, d_k, d_v, num_heads, dropout_rate) for _ in range(Nx)]
        )
        self.norm_final = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, X):
        X = self.positional_embedding(self.embedding(X))
        X = self.gpt_blocks(X)
        X = self.output(X)
        return X


# 3. Training model

**Train your model**

- Pretrained model for a few peochs
- Log training/validation loss and compute **perplexity**.
- Save checkpoints and final model.

**Generate text samples:**

- Use your trained model to generate coherent text related to your chosen domain.
- Show 3–5 examples with different prompts.
- Optionally experiment with **temperature, top-k, top-p sampling**

In [78]:
EPOCHS = 5
LR = 1e-3
device = 'cpu'
model = MiniGPT(
    N_LAYERS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    vocab_size=VOCAB_SIZE,
    dropout_rate=DROPOUT_RATE,
    context_size=1024,
)
optimizer = torch.optim.Adam(model.parameters(), lr = LR)
criterion = nn.CrossEntropyLoss()

def perplexity(cross_entropy_loss):
    return torch.exp(cross_entropy_loss)

def train_step(data_loader, model, optimizer):
    model.train()
    total_loss = 0
    total_perplexity = 0
    for inputs, labels in data_loader:
        inputs = inputs.to(device = device)
        labels = labels.to(device = device)

        raw_logits = model(inputs)
        loss = criterion(raw_logits, labels)

        total_loss += loss.item()
        # Add perplexity metric or BLEU here
        total_perplexity += perplexity(loss)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return total_loss / len(data_loader), total_perplexity / len(data_loader)


def eval_step(data_loader, model):
    model.eval()
    total_perplexity = 0
    with torch.inference_mode():
        total_loss = 0
        for inputs, labels in data_loader:
            inputs = inputs.to(device=device)
            labels = labels.to(device=device)

            raw_logits = model(inputs)
            loss = criterion(raw_logits, labels)
            total_loss += loss.item()
            total_perplexity += perplexity(loss)
            # Add perplexity metric or BLEU here
        return total_loss / len(data_loader), total_perplexity / len(data_loader)

In [77]:
a = torch.randint(0, 3, size = (10, ))
b = torch.randn(10, 3)

perplexity(criterion(b, a))

tensor(5.8388)

In [ ]:
train_losses = []
train_perplexities = []

val_losses = []
val_perplexities = []
for epoch in range(EPOCHS):
    train_loss, train_perplexity = train_step(train_loader, model, optimizer)
    val_loss, val_perplexity = eval_step(val_loader, model)
    train_losses.append(train_loss)
    train_perplexities.append(train_perplexity)
    val_losses.append(val_loss)
    val_perplexities.append(val_perplexity)
    # Logging model performance
    print(
        f"""
            Epoch: [{epoch}|{EPOCHS}]:     
            Train loss = {train_loss} - Train perplexity = {train_perplexity}
            Validation loss = {val_loss} - Validation perplexity = {val_perplexity}
            """
    )
    

In [79]:
# Plot model Performance
import matplotlib.pyplot as plt
plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

plt.figure()
plt.plot(train_perplexities, label="Train perplexity")
plt.plot(val_perplexities, label="Val perplexity")
plt.xlabel("Epoch")
plt.ylabel("Perplexity")
plt.title("Validation Perplexity")
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# TOMORROW: CONSTRUCT TRAINING AND TESTING DATASET TO RUN: USE TOKENIZER of GPT 

In [ ]:
def generate(model, top_k, temperature, top_p_sampling):
    pass


## **4. Evaluation:**

- Report quantitative results (loss, perplexity).
- Qualitative evaluation: human judgment of coherence and relevance.